# The Indexing Pipeline

In the previous notebook, we built a minimal end-to-end RAG system.

We saw this:

```text
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Vector Search
    ↓
Context
    ↓
LLM
    ↓
Answer
```
That gave us the basic mechanics.

Now we're going to slow down and investigate the **first half of that system**.

> **How does a collection of raw documents become a searchable knowledge base?**

That is the job of the **indexing pipeline**.

---
**What we'll learn**

By the end of this tutorial, you'll understand:

- What indexing means in RAG
- Why indexing happens separately from querying
- How documents move through the indexing pipeline
- Why ingestion and chunking are separate concerns
- What metadata should accompany a chunk
- Where embeddings fit into indexing
- What a vector index actually contains
- How document updates affect an index
- Where indexing pipelines commonly fail

Our target architecture is:

```text
Raw Documents
      ↓
Document Ingestion
      ↓
Normalization
      ↓
Chunking
      ↓
Metadata
      ↓
Embedding
      ↓
Indexing
      ↓
Searchable Knowledge Base

---
## 1. What Does "Indexing" Mean?

When we hear index, we might think of the index at the back of a book.

If a book contains hundreds of pages, its index helps us quickly locate information.

A RAG index serves a similar purpose.

Instead of searching through every document from scratch every time a user asks a question, we prepare the documents in advance so that relevant information can be found efficiently.

The basic idea is:

```text
                 WITHOUT INDEXING

User Query
    ↓
Search every document
    ↓
Find relevant information
```

versus

```text
                  WITH INDEXING

Documents
    ↓
Prepare and index
    ↓
Searchable Knowledge Base
            ↑
            │
        User Query
```

The second approach is what makes retrieval practical at scale.

---
## 2. Indexing Happens Before Retrieval

This is one of the most important distinctions in RAG.

There are two different moments:

**Indexing time**

We prepare the knowledge.

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embedding
    ↓
Indexing
```

**Query time**
We use the knowledge.

```text
User Query
    ↓
Retrieval
    ↓
Relevant Chunks
    ↓
Context
    ↓
LLM
```

So the complete system looks like:

```text
                 INDEXING TIME
                      │
                      ▼
                  Documents
                      │
                      ▼
                  Ingestion
                      │
                      ▼
                   Chunking
                      │
                      ▼
                  Embeddings
                      │
                      ▼
                   Indexing
                      │
                      ▼
                Knowledge Base
                      │
                      │
                      │
                   QUERY TIME
                      │
                      ▼
                  User Query
                      │
                      ▼
                   Retrieval
                      │
                      ▼
                  Reranking
                      │
                      ▼
                 Context
                      │
                      ▼
                     LLM
                      │
                      ▼
                   Answer
```

The indexing pipeline can be computationally expensive.

That's okay.

We don't necessarily have to perform it every time someone asks a question.

---
## 3. The Raw Document Is Not the Final Representation

Suppose we start with:

```text
refund_policy.pdf
```

The file itself isn't necessarily the thing we want to search directly.

We need to transform it.

```text
refund_policy.pdf
        ↓
    Extract content
        ↓
Structured representation
        ↓
      Chunks
        ↓
     Embeddings
        ↓
       Index
```

Each transformation gives us a representation better suited to the next stage.

This is a recurring pattern in data engineering:

> **Don't confuse the original data format with the representation needed by the system processing it.**

A PDF is optimized for displaying a document.

A vector index is optimized for retrieving information.

They serve different purposes.

---
## 4. Stage One — Document Ingestion

The first stage is ingestion.

We take a source document and extract the information we need.

For example:

```text
PDF
 ↓
PDF parser
 ↓
Text
 + 
Structure
 +
Metadata
```

A useful representation might contain:

```python
{
    "document_id": "refund-policy",
    "text": "...",
    "metadata": {
        "filename": "refund_policy.pdf",
        "page_count": 8
    }
}
```

At this point, we haven't created embeddings.

We haven't performed vector search.

We've simply converted the source into something our pipeline can process.

We'll explore ingestion much more deeply in **Module 3 — Document Ingestion**.

---
## 5. Stage Two — Normalization

Extracted content often contains noise.

For example, a parser might produce:

```text
REFUND POLICY

Page 1

Customers may request refunds...

Page 2

REFUND POLICY

Customers must return...
```

Some of that information may be useful.

Some may be repeated headers or footers.

Normalization is where we can clean and standardize the representation.

Potential operations include:

- Normalizing whitespace
- Removing repeated headers
- Removing unwanted footers
- Fixing encoding issues
- Normalizing line breaks
- Standardizing metadata

But there is an important warning:

> **Cleaning is not automatically an improvement.**

If we remove information that actually carries meaning, we've made the dataset worse.

This is one reason we'll inspect our data after every major transformation.

---
## 6. Stage Three — Chunking

Now we need to determine the units we actually want to retrieve.

A large document might become:

```text
Document
│
├── Chunk 1
├── Chunk 2
├── Chunk 3
├── Chunk 4
└── Chunk 5
```

Each chunk should ideally represent a coherent piece of information.

For example:

```text
Refund Policy

Customers may request refunds within 30 days of purchase.

Products must be returned in their original condition.
```

could become:

```text
Chunk 1:
Customers may request refunds within 30 days of purchase.

Chunk 2:
Products must be returned in their original condition.
```

But this isn't necessarily the best way to chunk every document.

A better chunk might preserve related information:

```text
Chunk 1:
Refund Policy

Customers may request refunds within 30 days of purchase.
Products must be returned in their original condition.
```

This is why chunking deserves its own module.

We'll investigate different strategies later.

---
## 7. Chunking Changes Retrieval

This is a subtle but important point.

Imagine the user asks:

> "What condition must a product be in for a refund?"

If our chunk contains:

```text
Customers may request refunds within 30 days.
```

but the condition appears in another chunk:

```text
Products must be returned in their original condition.
```

retrieval might find one without the other.

Our chunking decision has therefore affected retrieval quality.

This means:

```text
Chunking
    ↓
Retrieval
    ↓
Generation
```

is not independent.

A decision made during indexing can affect the final answer much later.

---
## 8. Stage Four — Metadata

Now let's attach information about where each chunk came from.

Instead of storing:

```python
{
    "text": "Customers may request refunds within 30 days."
}
```

we can store:

```python
{
    "chunk_id": "refund-policy-001",
    "document_id": "refund-policy",
    "text": "Customers may request refunds within 30 days.",
    "metadata": {
        "source": "refund_policy.pdf",
        "page": 4,
        "section": "Refund Eligibility"
    }
}
```

This gives us much more than search.

Metadata can later support:

- Filtering
- Citations
- Provenance
- Access control
- Document management
- Debugging
- Analytics

So metadata isn't an afterthought.

It is part of the representation we're building.

---
## 9. Stage Five — Embeddings

Now we transform the chunk into a vector.

```text
Chunk
  ↓
Embedding Model
  ↓
Vector
```

For example:

```text
"Customers may request refunds within 30 days."
                       ↓
                  Embedding
                       ↓
        [0.021, -0.184, 0.731, ...]
```

Our record now conceptually looks like:

```python
{
    "chunk_id": "refund-policy-001",
    "document_id": "refund-policy",
    "text": "Customers may request refunds within 30 days.",
    "embedding": [...],
    "metadata": {
        "source": "refund_policy.pdf",
        "page": 4,
        "section": "Refund Eligibility"
    }
}
```

This is approaching what we actually want to put into a retrieval system.

---
## 10. Stage Six — Indexing

Now we need to make those vectors searchable.

This is where a vector database or vector index comes in.

Conceptually:

```text
Chunk
 +
Embedding
 +
Metadata
     ↓
Vector Index
```

For our eventual architecture, we'll use Qdrant.

But notice something important.

Qdrant isn't responsible for:

```text
PDF parsing
Chunking
Choosing the embedding model
Understanding document structure
```

Those are responsibilities of earlier stages.

Qdrant's job is primarily to provide infrastructure for storing and searching vector representations, along with associated payload/metadata.

This separation of responsibilities will become important as the system grows.

---
## 11. What Actually Gets Indexed?
This is worth making explicit.

We don't just have:

```text
Vector
```

We have something closer to:

```text
┌─────────────────────────────────────────┐
│                RECORD                   │
├─────────────────────────────────────────┤
│ Chunk ID                                 │
│ Document ID                              │
│ Embedding                                │
│ Text                                     │
│ Metadata                                 │
│ Provenance                               │
└─────────────────────────────────────────┘
```

The exact storage design will depend on our vector database and architecture.

But conceptually, retrieval needs both:

**Representation**

```text
Embedding
```

and:

**Identity/context**

```text
Chunk
Document
Page
Section
Source
Permissions
```

Without the second category, we would have a difficult time explaining where retrieved information came from.

---
## 12. Let's Build the Representation

Let's make this concrete with Python.

We'll start with a simple document:

In [1]:
document = {
    "document_id": "refund-policy",
    "text": """
    Refund requests can be submitted within 30 days of purchase.
    Products must be returned in their original condition.
    Refunds are normally processed within 7 business days after approval.
    """,
    "metadata": {
        "source": "refund_policy.txt"
    }
}

Now create chunks:

In [2]:
def chunk_text(text, chunk_size=20):
    words = text.split()

    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

In [3]:
chunks = chunk_text(document["text"])

chunks

['Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are',
 'normally processed within 7 business days after approval.']

The exact output isn't the important part.

What's important is that:

```text
One document
     ↓
Multiple retrievable units
```

---
## 13. Attach Metadata

Now we'll preserve the document identity:

In [4]:
records = []

for index, chunk in enumerate(chunks):
    records.append({
        "chunk_id": f'{document["document_id"]}-{index}',
        "document_id": document["document_id"],
        "text": chunk,
        "metadata": document["metadata"]
    })

Inspect the result:

In [5]:
records

[{'chunk_id': 'refund-policy-0',
  'document_id': 'refund-policy',
  'text': 'Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are',
  'metadata': {'source': 'refund_policy.txt'}},
 {'chunk_id': 'refund-policy-1',
  'document_id': 'refund-policy',
  'text': 'normally processed within 7 business days after approval.',
  'metadata': {'source': 'refund_policy.txt'}}]

---
## 14. Generate Embeddings

### Load the embedding model

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
texts = [record["text"] for record in records]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

Attach them:

In [9]:
for record, embedding in zip(records, embeddings):
    record["embedding"] = embedding

Now inspect one:

In [10]:
records[0]

{'chunk_id': 'refund-policy-0',
 'document_id': 'refund-policy',
 'text': 'Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are',
 'metadata': {'source': 'refund_policy.txt'},
 'embedding': array([-7.81721845e-02,  3.56085540e-04,  5.37630990e-02,  2.64655910e-02,
        -2.92040892e-02, -2.91469749e-02,  4.00221497e-02,  3.43861021e-02,
        -4.50263768e-02,  1.96401253e-02, -6.02872157e-03,  1.41966725e-02,
        -2.71941703e-02,  4.94146273e-02, -2.59420834e-03,  5.04469126e-02,
         4.37017083e-02, -8.23119059e-02, -3.58429998e-02,  9.08553973e-03,
        -1.78538857e-03,  1.58453751e-02, -3.98171097e-02, -1.28609026e-02,
        -1.48760369e-02, -1.25570577e-02, -1.71105824e-02,  2.63265707e-03,
        -8.22781622e-02, -1.57202169e-01,  3.03783510e-02,  1.81793096e-03,
        -6.61692619e-02,  4.58512595e-03,  2.47048363e-02, -3.56504060e-02,
        -1.07728764e-02, -2.47236490e-02, -1.2833301

---
## 15. The Transformation We've Built

We started with:

```text
refund_policy.txt
```
and ended with:
```text
[
    {
        chunk_id,
        document_id,
        text,
        metadata,
        embedding
    },
    ...
]
```
The entire transformation is:
```text
             RAW DATA
                 │
                 ▼
             Document
                 │
                 ▼
              Parsing
                 │
                 ▼
            Normalization
                 │
                 ▼
              Chunking
                 │
                 ▼
             Metadata
                 │
                 ▼
             Embedding
                 │
                 ▼
              Indexing
                 │
                 ▼
          SEARCHABLE DATA
```
That is the indexing pipeline.

---
## 16. Indexing Is Not a One-Time Event

There's another important consideration.

Documents change.

Suppose the company changes its refund policy:

```text
Old:
Refunds within 30 days.

New:
Refunds within 60 days.
```

We now have to update the knowledge base.

This introduces questions such as:

- How do we detect changed documents?
- Do we delete the old chunks?
- Do we create new chunks?
- How do we prevent duplicate versions?
- How do we track document versions?
- What happens to embeddings?
- How do we handle deleted documents?

A production indexing pipeline therefore needs to think about **document lifecycle**, not just initial ingestion.

---
17. A Better Mental Model: Document Lifecycle

Instead of:
```text
Document → Index
```
think:
```text
             DOCUMENT LIFECYCLE

Created
   ↓
Ingested
   ↓
Parsed
   ↓
Chunked
   ↓
Embedded
   ↓
Indexed
   ↓
Updated
   ↓
Re-indexed
   ↓
Archived / Deleted
```
This becomes especially important when building systems where information changes frequently.

---
## 18. Where Indexing Can Fail

Let's look at some failure modes.

**Extraction failure**
```text
PDF
 ↓
Parser
 ↓
Missing text
```
The information never reaches retrieval.

---
**Chunking failure**
```text
Important information
        ↓
Split across unrelated chunks
        ↓
Retrieval loses context
```

---
**Metadata failure**
```text
Chunk
 ↓
Embedding
 ↓
Index
```
but we lose the page number.

Now we may be able to retrieve the information but cannot reliably tell the user where it came from.

---
**Duplicate indexing**

Suppose we accidentally index the same document three times:
```text
document_v1
document_v1
document_v1
```
Now retrieval may return duplicates and waste context.

---
**Stale data**
The source document changes:
```text
Old policy → New policy
```
but the old chunks remain in the index.

The RAG system may retrieve outdated information.

---
## 19. The Indexing Pipeline Is a Data Pipeline

This is perhaps the biggest conceptual takeaway from this notebook.

It is tempting to think about RAG as an LLM application.

But the indexing side looks much more like a **data engineering pipeline**:
```text
Source Data
    ↓
Extraction
    ↓
Transformation
    ↓
Enrichment
    ↓
Vectorization
    ↓
Storage
```
The LLM only appears much later.

This is why building good RAG systems requires more than prompt engineering.

You are building an **information processing system**.

---
## 20. Our Production Direction

Eventually, our indexing architecture will look more like:
```text
                    DOCUMENT SOURCES
                           │
                           ▼
                  Document Ingestion
                           │
                           ▼
                    Normalization
                           │
                           ▼
                 Structure Extraction
                           │
                           ▼
                  Structure-Aware
                      Chunking
                           │
                           ▼
                       Metadata
                           │
                           ▼
                     Embeddings
                           │
                           ▼
                    ┌──────────────┐
                    │    Qdrant    │
                    │              │
                    │ Vector Index │
                    │   + Payload  │
                    └──────────────┘
                           │
                           ▼
                    Searchable
                    Knowledge Base
```
We're **not** implementing all of that today.

We're building toward it.

---
## 21. Why We're Starting Simple

Our baseline is intentionally:
```text
TXT
 ↓
Simple chunks
 ↓
Embedding
 ↓
In-memory vectors
```
That's not because production RAG is this simple.

It's because we want to understand each transformation before introducing infrastructure.

The progression will be:
```text
Simple
  ↓
Understand
  ↓
Measure
  ↓
Improve
  ↓
Measure again
```
That approach will make the later architecture much easier to reason about.

---
### Key Takeaways

The indexing pipeline prepares external knowledge for retrieval.

The basic flow is:
```text
Documents
    ↓
Ingestion
    ↓
Normalization
    ↓
Chunking
    ↓
Metadata
    ↓
Embeddings
    ↓
Indexing
    ↓
Knowledge Base
```
The most important ideas are:

1. **Indexing happens before users query the system.**
2. A raw document is not the same thing as a searchable representation.
3. Chunking determines the units that retrieval can return.
4. Metadata should remain attached to chunks.
5. Embeddings provide a numerical representation useful for semantic retrieval.
6. A vector database provides infrastructure for storing and searching those representations.
7. Document updates require an indexing strategy.
8. Indexing is fundamentally a **data and information processing pipeline**.
9. Errors introduced during indexing can propagate all the way to the final generated answer.
